# transformers 라이브러리 공부

```
한 줄 요약: transformers 라이브러리에 대한 이해를 위한 공부
동기: AI-Augmented Surveys, SubPOP 같은 논문에서는 생성을 통한 학습이 아니기 때문에 학습 과정이 궁금해져 코드로 이를 이해하고자 함.
```

### AI-Augmented Surveys

```
해당 논문에서는 survey response prediction 태스크를 수행하여 다음과 같은 문제를 해결하고자 합니다.

1. missing value imputation
2. retrodiction (period 상황)
3. unasked question prediction

이때 사용하는 방법은 다음과 같습니다.

1. response embedding(4096-d to 50-d MLP), respondent embedding, period embedding
2. 세 임베딩을 DCN으로 학습

response embedding은 LLM을 통해 추출합니다.
따라서 코드를 통해 LLM에서 embedding을 추출하는 방법에 대해 이해하고자 하였습니다.
```

In [1]:
from transformers import AutoTokenizer, AutoModel
import torch

PROMPT = ("Below is an instruction that describes a task. Write a response that appropriately "
          "completes the request. ### Instruction: {q} ### Response: ")

question = "Spouse ever work as long as a year?"

tokenizer = AutoTokenizer.from_pretrained("chavinlo/alpaca-native")
model = AutoModel.from_pretrained("chavinlo/alpaca-native", dtype=torch.float16)
model.to("mps")
# device는 맥북이라 mps로

enc = tokenizer(PROMPT.format(q=question), return_tensors="pt").to("mps")
print(enc["input_ids"].shape) # [1,38]

h = model(**enc)
print(h[0].shape)             # [1, 38, 4096]

/opt/anaconda3/envs/survey_prediction/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `leg

torch.Size([1, 39])
torch.Size([1, 39, 4096])


```
DCN을 활용했다는 점에서 text로부터 얻은 semantic information을 활용하고 적은 추론 latency를 가지는 구조이기 때문에 흥미로웠습니다.
또한 논문에서의 Figure A2에서 임베딩 공간상 구조가 잘 잡혀있는 것이 보여서 흥미로웠습니다.
```

### SubPOP

```
해당 논문에서는 pilot survey를 사람이 아닌 LLM을 대상으로 하는 문제를 풀고자 하고,
결국엔 다른 subpopulation의 response distibution을 학습합니다.
이때 사용하는 방법은 다음과 같습니다.

1. forward KL: 소수 의견도 잘 학습
2. LoRA
3. metric으로 Wasserstein Distance 사용

논문에서는 다른 survey 데이터(ATP --> GSS)에서도 SubPOP-FT 방식이 좋은 성능을 보인다고 합니다.
정확히 model의 out 중에서 특정 토큰의 로짓만을 사용해야 하기 때문에 이를 코드를 통해 구현하는 방법을 이해하였습니다.
코드를 작성할 때 논문의 공식 레포지토리를 참고하였습니다.
```

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")
model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-v0.1", dtype=torch.float16)
model.to("mps")

example = (
    "Question: In politics today, do you consider yourself a Democrat, Republican, or Independent?\n"
    "Answer: Republican\n\n"
    "Question: Do you think the federal government should do more to regulate businesses, "
    "or is it already doing too much?\n"
    "A. Should do more\n"
    "B. Doing about the right amount\n"
    "C. Already doing too much\n"
    "D. Refused\n"
    "Answer:"
) # example은 클로드가 만들어준 예시
# 원본 논문에서의 코드 역시 'A'와 같은 걸 붙이도록 하는데,
# 여기서 " A"라고 응답하면 첫번째 응답을 하는 걸 LLM이 알고 있기 때문에 'A' 토큰의 logit을 가져오면 됨
# 논문의 깃허브의 subpop/subpop/utils/survey_utils.py 에서 generate_mcq 함수에 구성하는 게 나옴

enc = tok(example, return_tensors="pt").to("mps")
out = model(**enc)
logits = out.logits.float().contiguous()   # 1, 82, 32000
print(f"모델 logits: {logits.shape}")
print(f"vocab size: {tok.vocab_size}")     # 32000

# 여기서 마지막 토큰과 target_ids가 필요하다.
print(tok("A")) # 330
# B, C, D = 365, 334, 384
ABCD = [330, 365, 334, 384]

prob = F.softmax(logits[:, -1, ABCD], dim = -1) # vocab 중 ABCD 중에서 softmax
print(prob)    # 0.3571, 0.2613, 0.2892, 0.0924

# 이후에는 prob을 실제 설문 응답 비율에 맞추는 kl loss로 학습

Loading checkpoint shards: 100%|██████████| 2/2 [00:16<00:00,  8.00s/it]


모델 logits: torch.Size([1, 82, 32000])
vocab size: 32000
{'input_ids': [1, 330], 'attention_mask': [1, 1]}
tensor([[0.3571, 0.2613, 0.2892, 0.0924]], device='mps:0',
       grad_fn=<SoftmaxBackward0>)


```
transformers의 작동을 파악하기 위함이라서 부분적인 구현만 진행해봤습니다.
```

```
역문항 실험
LLM은 A, B, C, D 중에 고르라면 먼저 나온 선지를 고르는 경향이 있기 때문에 방금의 코드를 바꿔 이를 확인하고자 합니다.
순서 변화: ABCD --> CBAD
```

In [3]:
example = (
    "Question: In politics today, do you consider yourself a Democrat, Republican, or Independent?\n"
    "Answer: Republican\n\n"
    "Question: Do you think the federal government should do more to regulate businesses, "
    "or is it already doing too much?\n"
    "A. Already doing too much\n"
    "B. Doing about the right amount\n"
    "C. Should do more\n"
    "D. Refused\n"
    "Answer:"
) # example은 클로드가 만들어준 예시
# 원본 논문에서의 코드 역시 'A'와 같은 걸 붙이도록 하는데,
# 여기서 " A"라고 응답하면 첫번째 응답을 하는 걸 LLM이 알고 있기 때문에 'A' 토큰의 logit을 가져오면 됨
# 논문의 깃허브의 subpop/subpop/utils/survey_utils.py 에서 generate_mcq 함수에 구성하는 게 나옴

enc = tok(example, return_tensors="pt").to("mps")
out = model(**enc)
logits = out.logits.float().contiguous()

prob = F.softmax(logits[:, -1, ABCD], dim = -1)
print(prob)

# 이후에는 prob을 실제 설문 응답 비율에 맞추는 kl loss로 학습

tensor([[0.6917, 0.1580, 0.0922, 0.0581]], device='mps:0',
       grad_fn=<SoftmaxBackward0>)


```
역문항까지 고려하면 Republican은 기업 규제에 대해 Too Much라고 답할 것이라는 게 LLM이 학습한 지식이라는 것을 알 수 있습니다.
```